# Pilot 1

- Upbringing (Good/Bad) × Belief (Activist/Bigot) × Action (Help/Harm)
- Responsibility for Action (–6 to +6)
- N = 399

## Set Up Notebook

In [ ]:
##########
# IMPORT #
##########

# Standard library imports
import warnings

# Third-party imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns
import statsmodels.api as sm
from patsy.contrasts import Sum
from statsmodels.formula.api import ols


#################
# CONFIGURATION #
#################

# Set global plotting style
sns.set_theme(style = "whitegrid")


##########
# LABELS #
##########

# Measure
labels_measure = {
    'measure_action': 'Responsibility for Action'
}

# Gender
labels_gender = {
    1: 'Female',
    2: 'Male',
    3: 'Non-binary',
    4: 'Prefer not to say'
}


#################
# VISUALIZATION #
#################

# Colors
palette_main = {
    "Good":     "#0072B2", "Bad":   "#D55E00",
    "Activist": "#56B4E9", "Bigot": "#E69F00",
    "Help":     "#F0E442", "Harm":  "#882255"
}

## Transform Data

In [ ]:
# Load data
df = pd.read_csv('blame_praise_self_pilot_1_data.csv')

# Define factors
factors_map = {
     5: {'Upbringing': "Bad",  'Belief': "Activist", 'Action': "Help"},
     6: {'Upbringing': "Bad",  'Belief': "Activist", 'Action': "Harm"},
     7: {'Upbringing': "Bad",  'Belief': "Bigot",    'Action': "Help"},
     8: {'Upbringing': "Bad",  'Belief': "Bigot",    'Action': "Harm"},
    13: {'Upbringing': "Good", 'Belief': "Activist", 'Action': "Help"},
    14: {'Upbringing': "Good", 'Belief': "Activist", 'Action': "Harm"},
    15: {'Upbringing': "Good", 'Belief': "Bigot",    'Action': "Help"},
    16: {'Upbringing': "Good", 'Belief': "Bigot",    'Action': "Harm"}
}

# Reshape wide to long
indices = [5, 6, 7, 8, 13, 14, 15, 16]
long_data = []

for _, row in df.iterrows():
    p_id = row['ID']
    
    for i in indices:
        col_action = f'homophobia.{i}-pAction'
        
        if col_action in df.columns and pd.notna(row[col_action]):
            score_action = pd.to_numeric(row[col_action], errors = 'coerce') - 7
            
            entry = {
                'ID': p_id,
                'measure_action': score_action
            }
            entry.update(factors_map[i])
            
            long_data.append(entry)

df_long = pd.DataFrame(long_data)
print(f"Transformation complete ({len(df_long)} observations).")

## Calculate Sociodemographics

In [ ]:
# Prepare variables
df['Gender_Labeled'] = df['Gender'].map(labels_gender)
df['Age'] = pd.to_numeric(df['Age'], errors = 'coerce')

# Calculate sociodemographics and display results
for col in ['Gender_Labeled']:
    print(f"\n{col}:")
    display(df[col].value_counts().to_frame('Frequency'))

print("\nAge:")
display(df['Age'].describe().to_frame('Value').round(3))

print("\nParticipants:")
total_participants = df_long['ID'].nunique()
display(f"N = {total_participants}")

## Calculate Descriptive Statistics

In [ ]:
# Define group factors and dependent variables
group_factors = ['Upbringing', 'Belief', 'Action']
dependent_var = ['measure_action']

# Calculate descriptive statistics
descriptive_stats = (
    df_long
    .groupby(group_factors)[dependent_var]
    .agg(['mean', 'std', 'count'])
    .round(3)
)

# Display results
display(descriptive_stats)

## Perform ANOVA

In [ ]:
# Define formula
formula = "measure_action ~ C(Upbringing, Sum) * C(Belief, Sum) * C(Action, Sum)"

# Fit model
model = ols(formula, data = df_long).fit()

# Run ANOVA
aov_table = sm.stats.anova_lm(model, typ = 3)

# Calculate effect sizes
aov_table['partial_eta_sq'] = aov_table['sum_sq'] / (aov_table['sum_sq'] + aov_table.loc['Residual', 'sum_sq'])

# Display results
display(aov_table.round(3))

## Generate Histograms

In [ ]:
# Set condition labels
df_long['Condition'] = (df_long['Upbringing'] + " / " +
                        df_long['Belief'] + " / " +
                        df_long['Action']
                       )

# Generate graph
for dv in dependent_var:
    g = sns.displot(data     = df_long,
                    x        = dv,
                    col      = "Condition",
                    col_wrap = 4,
                    kind     = "hist",
                    discrete = True,
                    shrink   = 0.8,
                    height   = 3,
                    aspect   = 1.2,
                    color    = "#0072B2"
                   )
    
    # Set axis labels and subplot titles
    g.set_axis_labels("Value", "Count")
    g.set_titles("{col_name}")
    
    # Limit x-axis
    for ax in g.axes.flat:
        ax.set_xticks(range(-6, 7))
        ax.set_xlim(-6.5, 6.5)
    
    # Add title
    plt.subplots_adjust(top = 0.8)
    g.fig.suptitle(f'{labels_measure[dv]}', fontsize = 16)
    
    # Export graph
    filename = 'blame_praise_self_pilot_1_histograms.png'
    g.savefig(filename, dpi = 300, bbox_inches = 'tight')
    plt.show()
    print(f"Graph saved as '{filename}'.")

## Generate Bar Plot

In [ ]:
# Generate graph
g = sns.catplot(data    = df_long,
                x       = "Upbringing",
                y       = "measure_action",
                hue     = "Belief",
                col     = "Action",
                kind    = "bar",
                palette = palette_main,
                capsize = 0.05,
                height  = 5
               )

# Set axis labels and subplot titles
g.set_axis_labels("Upbringing", "Mean (–6 to +6)")
g.set_titles("{col_name}")

# Add horizontal zero lines and limit y-axis
for ax in g.axes.flat:
    ax.axhline(0, color = 'black', lw = 1)
    ax.set_ylim(-6, 6)

# Add main title
plt.subplots_adjust(top = 0.85)
g.fig.suptitle('Responsibility for Action', fontsize = 16)

# Export graph
filename = 'blame_praise_self_pilot_1_bar_plot.png'
g.savefig(filename, dpi = 300, bbox_inches = 'tight')
plt.show()
print(f"Graph saved as '{filename}'.")